# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue: 160,883 published pages from March 2026, ranked by opportunity_score (CTR gap
weighted by log-impressions, to prevent a handful of huge-traffic pages from dominating —
a fix from an earlier version of this scoring that over-weighted raw impressions).

Reason codes, in priority order:
1. underperforming_for_position (68,803 pages) — real CTR gap, meaningful traffic (≥100
   impressions). Review these first.
2. gap_but_low_volume (57,359 pages) — a gap exists but on too little traffic to trust the
   signal. Watchlist, not action list.
3. meeting_or_beating_expected_ctr (34,721 pages) — no action indicated.

What "first" means here: highest opportunity_score within underperforming_for_position —
these combine the largest relative CTR shortfall with meaningful traffic, so acting on them
touches the most impressions per unit of editorial effort.

In [1]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

df = con.sql(f"""
    SELECT f.client_hash_id, f.content_hash_id, f.report_date,
           d.content_type, d.main_intent,
           f.gsc_avg_position, f.gsc_impressions, f.gsc_clicks
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id AND f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS FALSE
      AND f.gsc_impressions > 0
""").df()

# aggregate to one row per page across the month (not page-day)
agg = df.groupby(["client_hash_id", "content_hash_id", "content_type", "main_intent"], as_index=False).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean"),
)
agg["ctr"] = agg["clicks"] / agg["impressions"]

bins = [0, 3, 5, 10, 20, 50, np.inf]
labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]
agg["position_bucket"] = pd.cut(agg["avg_position"], bins=bins, labels=labels)

bench = agg.groupby("position_bucket", observed=True).apply(
    lambda g: g["clicks"].sum() / g["impressions"].sum(), include_groups=False
)
agg["expected_ctr"] = agg["position_bucket"].map(bench).astype(float)
agg["ctr_gap"] = agg["expected_ctr"] - agg["ctr"]

# FIX from the capstone: log-dampen impressions so a handful of huge pages don't dominate
agg["opportunity_score"] = agg["ctr_gap"] * np.log1p(agg["impressions"])

def reason_code(row):
    if row["ctr_gap"] <= 0:
        return "meeting_or_beating_expected_ctr"
    if row["impressions"] < 100:
        return "gap_but_low_volume"
    return "underperforming_for_position"

agg["reason_code"] = agg.apply(reason_code, axis=1)
ranked = agg.sort_values("opportunity_score", ascending=False).reset_index(drop=True)

print(f"Ranked {len(ranked):,} pages")
print(ranked["reason_code"].value_counts())
print(ranked[["content_hash_id", "position_bucket", "ctr", "expected_ctr", "impressions", "opportunity_score", "reason_code"]].head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked 160,883 pages
reason_code
underperforming_for_position       68803
gap_but_low_volume                 57359
meeting_or_beating_expected_ctr    34721
Name: count, dtype: int64
             content_hash_id position_bucket       ctr  expected_ctr  \
0   content_d61fc394d10cba41             1-3  0.000026      0.004052   
1   content_fc67675904376267             1-3  0.000299      0.004052   
2   content_8e1334d6356668e3             4-5  0.000007      0.003501   
3   content_306bc78dff1eb683             1-3  0.000433      0.004052   
4   content_66bf45eb0c5bb550             1-3  0.000041      0.004052   
5   content_b9acd1ebff7d34ff             1-3  0.000116      0.004052   
6   content_1d7764b642f7bb9f             1-3  0.000171      0.004052   
7   content_c46df0fa61530d86             1-3  0.000597      0.004052   
8   content_fa17add7836d36c3             1-3  0.000000      0.004052   
9   content_34a70fea29d15f24             4-5  0.000301      0.003501   
10  content_805fd45594ea23

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [2]:
top_bucket_share = ranked.head(1000)["position_bucket"].value_counts(normalize=True).round(3)
print("Position bucket share of top 1,000 flagged pages:")
print(top_bucket_share)

median_impressions_flagged = ranked[ranked["reason_code"]=="underperforming_for_position"]["impressions"].median()
print(f"\nMedian impressions among underperforming_for_position: {median_impressions_flagged:,.0f}")

Position bucket share of top 1,000 flagged pages:
position_bucket
1-3      0.696
4-5      0.246
6-10     0.035
11-20    0.023
21-50    0.000
51+      0.000
Name: proportion, dtype: float64

Median impressions among underperforming_for_position: 718


Who uses this: a content team deciding what to review this week, when they can't manually
audit hundreds of pages. Input to a human decision, not an automated action.

Where it stops being valid:
- The queue skews heavily toward top positions — 69.6% of the top 1,000 flagged pages sit in
  position 1-3, only 2.3% in 11-20 and effectively none below that. This is a real
  consequence of using an absolute CTR gap: top positions have a much higher expected CTR
  (0.41%) than lower ones (0.07% at 51+), so the same relative shortfall produces a larger
  absolute gap near the top. This queue is decision-support for "which top-ranking pages
  are leaving clicks on the table" — it is not a general-purpose "everything wrong with our
  content" list, and a team relying on it alone would systematically under-review
  mid-and-low-ranking pages that may have their own real problems.
- One month of data (March 2026), one snapshot. This says nothing about whether a page's
  gap is new, worsening, or already being addressed.
- The ctr_gap benchmark itself has an unexplained anomaly (11-20 slightly out-clicking 6-10,
  documented in the capstone) — pages in those two buckets are being scored against a
  benchmark curve with a known irregularity.
- Correlational, not diagnostic: a flagged page is worth a human look, not proof of what's
  wrong with it (title, snippet, SERP feature, intent mismatch all remain possible causes).
- median impressions among the "review these" bucket is 718 — moderate traffic, not the
  site's biggest pages typically, though the ranking is weighted to favor higher-traffic
  pages within that group.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

What a person must check before acting on any flagged page:
- Confirm the page is still live and intentionally published (is_deleted/is_published were
  filtered at the query level, but re-verify at review time — a month-old snapshot can lag).
- Check whether a SERP feature (featured snippet, People Also Ask, image pack) is visibly
  present for this page's main query — that alone can explain a CTR gap no content change
  will fix.
- Check whether this page's queries carry commercial vs. informational intent — a mismatch
  between page content and dominant search intent may need a content rewrite, not a
  metadata tweak.

What should never be automated from this list alone:
- Never auto-generate or auto-publish new titles/meta descriptions from this queue without
  human review — the queue identifies *that* a gap exists, not *why*, so an automated fix
  could target the wrong root cause.
- Never treat a low ctr_gap as "this page is fine" without checking impressions first —
  gap_but_low_volume pages may have a real problem that simply hasn't accumulated enough
  traffic to be statistically visible yet.
- Never use this queue to make claims about search engine behavior, ranking algorithm logic,
  or causal statements like "improving CTR will improve position" — this data is
  cross-sectional and associational only (see writing-honest-claims: no causal language
  without a controlled design).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Signals that this queue has gone stale and needs rebuilding, not just re-running:
- A new month's benchmark curve (expected_ctr per bucket) shifts meaningfully from March's
  values (e.g. 1-3 bucket moving far from ~0.41%) — algorithm or SERP-layout changes could
  invalidate the benchmark itself.
- The position-bucket concentration in the top 1,000 (currently 69.6% at 1-3) shifts sharply
  — could mean the underlying position distribution changed, or a scoring bug.
- gsc_data_available coverage (36.7% in March, per the data contract) drops significantly —
  less real signal means noisier gap estimates.
- More than ~3 months since the queue was last rebuilt — page content, rankings, and SERP
  layouts change; treat this as a monthly-refresh artifact, not a static one.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [3]:
from pathlib import Path

Path("work/outputs").mkdir(parents=True, exist_ok=True)
ranked.to_csv("work/outputs/action_playbook_march2026.csv", index=False)

summary = ranked["reason_code"].value_counts().reset_index()
summary.columns = ["reason_code", "count"]
summary.to_csv("work/outputs/action_playbook_summary.csv", index=False)

print("Exported:")
print("- work/outputs/action_playbook_march2026.csv:", len(ranked), "rows")
print("- work/outputs/action_playbook_summary.csv:", len(summary), "rows")
print(summary)

Exported:
- work/outputs/action_playbook_march2026.csv: 160883 rows
- work/outputs/action_playbook_summary.csv: 3 rows
                       reason_code  count
0     underperforming_for_position  68803
1               gap_but_low_volume  57359
2  meeting_or_beating_expected_ctr  34721


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous hashes used)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.